# 04 Schema and Data Types

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Define a DataFrame's schema yourself, know the common Spark data types, understand the trade-offs of schema inference, and convert types safely with <code>cast</code> and <code>try_cast</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Wrong types are one of the most common causes of broken pipelines: amounts stored as strings, dates that silently become null, IDs that overflow. In production you almost always <b>declare</b> the schema instead of letting Spark guess. Interviewers ask "why not use <code>inferSchema</code>?" all the time.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What is a schema?</b><br>
A schema is the list of columns with, for each one, a <b>name</b>, a <b>data type</b> and whether it is <b>nullable</b>.
<ul><li><code>StructType</code> is the whole schema (a list of fields)</li><li><code>StructField(name, type, nullable)</code> is one column</li><li>A DDL string such as <code>"id INT, name STRING"</code> is a shorter way to write the same thing</li><li>The schema lives in the DataFrame's metadata, so reading it costs nothing</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An insurer receives a daily claims file from a partner. If Spark infers <code>claim_amount</code> as a string because one row contains <code>"N/A"</code>, every downstream sum fails. With an explicit schema plus <code>try_cast</code>, the bad value becomes null, gets counted by a data quality check, and the pipeline keeps running.
</div>

## Common Spark data types

| Spark type | DDL name | Python value | Use it for |
|---|---|---|---|
| `StringType` | `STRING` | `str` | Text, codes, IDs with leading zeros |
| `IntegerType` | `INT` | `int` | Whole numbers up to about 2.1 billion |
| `LongType` | `BIGINT` | `int` | Large whole numbers, row counts, epoch millis |
| `DoubleType` | `DOUBLE` | `float` | Measurements and scientific values (approximate) |
| `DecimalType(p, s)` | `DECIMAL(p,s)` | `Decimal` | **Money**, where exact precision matters |
| `BooleanType` | `BOOLEAN` | `bool` | Flags |
| `DateType` | `DATE` | `datetime.date` | Calendar dates |
| `TimestampType` | `TIMESTAMP` | `datetime.datetime` | Date plus time, with time zone semantics |
| `ArrayType(t)` | `ARRAY<t>` | `list` | Lists such as tags or line items |
| `MapType(k, v)` | `MAP<k,v>` | `dict` | Key-value attributes |
| `StructType` | `STRUCT<...>` | `Row` / `dict` | Nested records such as an address |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
Use <code>DECIMAL</code> for money and <code>DOUBLE</code> for measurements. <code>0.1 + 0.2</code> is not exactly <code>0.3</code> in a <code>DOUBLE</code>.
</div>

## Syntax

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Option 1: StructType (explicit, good for programmatic schemas)
schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("name", StringType(), nullable=True),
])

# Option 2: DDL string (short and readable)
schema = "id INT NOT NULL, name STRING"

df = spark.createDataFrame(data, schema)
df = df.withColumn("id", df["id"].cast("bigint"))
```

## 1. Let Spark infer the schema

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a DataFrame without a schema and lets Spark guess the types from the Python values.<br><br>
<b>Why it matters</b><br>Inference is convenient, but you should know exactly what Spark guesses, because the guesses are often wider than you want.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>id</code> and <code>salary</code> become <code>long</code>, <code>rating</code> becomes <code>double</code>, <code>joined</code> becomes <code>date</code>. All columns are nullable.
</div>

In [0]:
# Python values carry their own types, so Spark can infer a schema from them.
from datetime import date

data = [
    (1, "Amal", 21000, 4.5, date(2023, 1, 15)),
    (2, "Senthil", 150000, 3.9, date(2021, 6, 1)),
    (3, "John", 18000, 4.1, date(2024, 3, 20)),
]

# Only column names are given, so Spark must infer each type.
df_inferred = spark.createDataFrame(data, ["id", "name", "salary", "rating", "joined"])

# Python int -> long, float -> double, date -> date. Everything is nullable.
df_inferred.printSchema()

## 2. Define the schema with StructType

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Declares each column's name, type and nullability with <code>StructType</code> and <code>StructField</code>.<br><br>
<b>Why it matters</b><br>This is the production pattern. You control the types, you document the data contract in code, and Spark doesn't need to inspect data to guess.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema with <code>id: integer (nullable = false)</code>, <code>salary: decimal(10,2)</code> and <code>joined: date</code>, then the 3 rows.
</div>

In [0]:
# Import the type classes we need.
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DecimalType, DoubleType, DateType
)
from decimal import Decimal

# A StructType is a list of StructFields: (name, type, nullable).
employee_schema = StructType([
    StructField("id", IntegerType(), nullable=False),     # must always have a value
    StructField("name", StringType(), nullable=True),
    StructField("salary", DecimalType(10, 2), nullable=True),  # exact money values
    StructField("rating", DoubleType(), nullable=True),
    StructField("joined", DateType(), nullable=True),
])

# DecimalType expects Python Decimal values, so we convert the salaries.
data_typed = [
    (1, "Amal", Decimal("21000.00"), 4.5, date(2023, 1, 15)),
    (2, "Senthil", Decimal("150000.00"), 3.9, date(2021, 6, 1)),
    (3, "John", Decimal("18000.00"), 4.1, date(2024, 3, 20)),
]

df_explicit = spark.createDataFrame(data_typed, employee_schema)
df_explicit.printSchema()
df_explicit.show()

## 3. Define the schema with a DDL string

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same kind of schema as a SQL-style string.<br><br>
<b>Why it matters</b><br>DDL strings are shorter and are the same syntax you use in <code>CREATE TABLE</code>, in <code>spark.read.schema(...)</code> and in <code>from_json</code>. Many teams prefer them for readability.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>product_id: integer</code>, <code>price: decimal(8,2)</code>, <code>tags: array&lt;string&gt;</code>, then 2 rows.
</div>

In [0]:
# The same idea as StructType, written as one string.
ddl_schema = "product_id INT, product STRING, price DECIMAL(8,2), tags ARRAY<STRING>"

products = [
    (101, "Laptop", Decimal("899.99"), ["electronics", "computers"]),
    (102, "Desk", Decimal("249.50"), ["furniture"]),
]

df_products = spark.createDataFrame(products, ddl_schema)
df_products.printSchema()
df_products.show(truncate=False)

# Every schema can be turned back into a DDL string, which is handy for documentation.
print(df_products.schema.simpleString())

## 4. Nested types: struct, array and map

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a DataFrame with a nested <code>address</code> struct, an <code>array</code> of phone numbers and a <code>map</code> of attributes, then reads nested fields.<br><br>
<b>Why it matters</b><br>JSON from APIs and event streams is nested. You will meet these types again in lesson 22 (reading JSON) and in the Bronze layer of every lakehouse.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A tree-shaped schema, then a table with <code>city</code>, the first phone number and the <code>tier</code> attribute pulled out of the nested columns.
</div>

In [0]:
from pyspark.sql import functions as F

customer_schema = """
    customer_id INT,
    address STRUCT<street: STRING, city: STRING, postcode: STRING>,
    phones ARRAY<STRING>,
    attributes MAP<STRING, STRING>
"""

customers = [
    (1, ("12 Anna Salai", "Chennai", "600002"), ["+91-900000001"], {"tier": "gold", "channel": "app"}),
    (2, ("5 Sheikh Zayed Rd", "Dubai", "00000"), ["+971-50000001", "+971-50000002"], {"tier": "silver"}),
]

df_customers = spark.createDataFrame(customers, customer_schema)

# printSchema() shows nested types as a tree.
df_customers.printSchema()

# Dot notation reads a struct field; [index] reads an array element; [key] reads a map value.
df_customers.select(
    "customer_id",
    F.col("address.city").alias("city"),
    F.col("phones")[0].alias("first_phone"),
    F.col("attributes")["tier"].alias("tier"),
).show()

## 5. Inspect a schema programmatically

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Loops over <code>df.schema.fields</code> to read each column's name, type and nullability.<br><br>
<b>Why it matters</b><br>Data quality checks and generic pipeline code often need to inspect a schema, for example to find all decimal columns or compare a new file against the expected contract.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One line per column, then the list of numeric columns: <code>['id', 'salary', 'rating']</code>.
</div>

In [0]:
from pyspark.sql.types import NumericType

# df.schema is a StructType. Its .fields are StructField objects.
for field in df_explicit.schema.fields:
    print(f"{field.name:<8} {field.dataType.simpleString():<14} nullable={field.nullable}")

# Find every numeric column (IntegerType, DecimalType, DoubleType all extend NumericType).
numeric_cols = [f.name for f in df_explicit.schema.fields if isinstance(f.dataType, NumericType)]
print("Numeric columns:", numeric_cols)

# Compare two schemas: useful as a quick contract check.
print("Same schema as inferred version?", df_explicit.schema == df_inferred.schema)

## 6. Changing types with `cast`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts string columns into proper types with <code>cast</code>, using both the type-object and the string forms.<br><br>
<b>Why it matters</b><br>Raw files usually arrive as strings. Casting is the first step of every Silver-layer transformation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All three target columns show their new types (<code>int</code>, <code>decimal(10,2)</code>, <code>date</code>) and values look the same.
</div>

In [0]:
# Everything arrives as text, as it would from a raw CSV.
raw = spark.createDataFrame(
    [("1", "21000.50", "2023-01-15"), ("2", "150000", "2021-06-01")],
    "id STRING, salary STRING, joined STRING",
)

typed = (
    raw
    .withColumn("id", F.col("id").cast("int"))                    # string form
    .withColumn("salary", F.col("salary").cast(DecimalType(10, 2)))  # type-object form
    .withColumn("joined", F.col("joined").cast("date"))
)

typed.printSchema()
typed.show()

## 7. Edge case: bad values, ANSI mode and `try_cast`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to cast a value that is not a number, first with <code>cast</code> and then with <code>try_cast</code>.<br><br>
<b>Why it matters</b><br>Serverless compute and Spark 4 run with <b>ANSI mode on</b>. In ANSI mode an invalid <code>cast</code> <b>raises an error</b> instead of silently returning null, which is safer but surprises people who learned older Spark. <code>try_cast</code> gives you the old "null on failure" behavior explicitly.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
In older Spark versions (and with ANSI mode off), <code>cast("int")</code> on <code>"N/A"</code> returns null without any error. Many online tutorials still assume that behavior.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first cell prints a <code>CAST_INVALID_INPUT</code> (or similar) error message. The second cell shows <code>null</code> for <code>N/A</code> and a count of 1 bad row.
</div>

In [0]:
dirty = spark.createDataFrame([("100",), ("250",), ("N/A",)], "amount STRING")

# Is ANSI mode on? On serverless and Spark 4 this is normally 'true'.
try:
    print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))
except Exception:
    print("ANSI mode setting not readable here")

# With ANSI mode on, casting 'N/A' to int fails when the action runs.
try:
    dirty.withColumn("amount_int", F.col("amount").cast("int")).show()
except Exception as e:
    print("cast failed:", type(e).__name__, "-", str(e).split("\n")[0][:150])

In [0]:
# try_cast returns null instead of failing. Available as a Column method in Spark 4.
safe = dirty.withColumn("amount_int", F.col("amount").try_cast("int"))
safe.show()

# The same thing in SQL syntax, which also works on older runtimes.
safe_sql = dirty.withColumn("amount_int", F.expr("try_cast(amount AS INT)"))

# Count the values that could not be converted: a simple data quality check.
bad_rows = safe_sql.filter(F.col("amount").isNotNull() & F.col("amount_int").isNull()).count()
print("Rows that failed conversion:", bad_rows)

## 8. Edge case: precision, overflow and nullability

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows three classic traps: floating-point rounding, decimal precision, and how Spark treats <code>nullable=False</code>.<br><br>
<b>Why it matters</b><br>These produce wrong numbers without obvious errors. They are favorite interview "gotchas".
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The double sum prints <code>0.30000000000000004</code> while the decimal sum prints <code>0.30</code>. Then creating a DataFrame with a null in a non-nullable column is rejected with an error (the exact message differs between classic compute and serverless).
</div>

In [0]:
# Floating point: approximate. Decimal: exact.
nums = spark.createDataFrame([(0.1, Decimal("0.10")), (0.2, Decimal("0.20"))], "d DOUBLE, m DECIMAL(5,2)")
nums.agg(F.sum("d").alias("double_sum"), F.sum("m").alias("decimal_sum")).show()

# nullable=False is enforced when we create a DataFrame from local data.
strict_schema = StructType([StructField("id", IntegerType(), nullable=False)])
try:
    spark.createDataFrame([(1,), (None,)], strict_schema).show()
except Exception as e:
    print("Rejected:", type(e).__name__, "-", str(e).split("\n")[0][:150])

## Under the hood

```
createDataFrame(data, schema)
        |
        v
  Schema stored in the DataFrame's metadata (StructType)
        |
        v
  printSchema() / df.schema / df.dtypes  -> read metadata only, no job
        |
        v
  cast / try_cast -> added to the logical plan as Cast expressions
        |
        v
  Action (show, count, write) -> Spark evaluates the casts row by row on executors
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> `printSchema()`, `schema`, `dtypes` never touch the data.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `cast` and `try_cast` add an expression to the plan.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> invalid casts only fail when an action runs, because of lazy evaluation.

When reading files, **inferring** a schema means Spark must read the data an extra time before the real job, which is slow on large data and can guess wrong. An explicit schema avoids that pass completely. You will measure this in lesson 21.

Run the cell below and look for `cast(...)` inside the plan.

In [0]:
# explain() prints the plan. Look for the Cast / try_cast expressions in the Project step.
typed.explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>CAST_INVALID_INPUT</code> error</b><br>
ANSI mode is on, and a value can't be converted. Use <code>try_cast</code> (and count the nulls it produces), or clean the value first.<br><br>
<b>⛔ Problem: money totals are slightly off</b><br>The column is <code>DOUBLE</code>. Use <code>DECIMAL(p, s)</code> for currency.<br><br>
**⛔ Problem: <code>DecimalType</code> column rejects Python floats**<br><code>createDataFrame</code> expects <code>decimal.Decimal</code> values for a <code>DecimalType</code> field. Convert them, or create the column as double and <code>cast</code> it.<br><br>
<b>⛔ Problem: dates become null or raise errors</b><br>The string format doesn't match the default <code>yyyy-MM-dd</code>. Use <code>to_date(col, "dd/MM/yyyy")</code> (lesson 15).<br><br>
<b>⛔ Problem: IDs lose their leading zeros</b><br>Postcodes and account numbers such as <code>00123</code> were cast to a number. Keep identifiers as <code>STRING</code>.<br><br>
**⛔ Problem: <code>Can not infer schema</code> for an empty list**<br>Spark can't guess types from no data. Always pass a schema.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a schema in Spark?</b><br>
The structure of a DataFrame: each column's name, data type and nullability, stored as a <code>StructType</code> of <code>StructField</code>s. It is metadata, so reading it is free.<br><br>

<b>🎤 2. Why would you define a schema explicitly instead of using <code>inferSchema</code>?</b><br>
Three reasons. Performance, because inference needs an extra pass over the data. Correctness, because inference can guess wrong, for example treating IDs as numbers. And stability, because the schema becomes a documented contract that doesn't change when a file changes.<br><br>

<b>🎤 3. What is the difference between <code>StructType</code> and a DDL string?</b><br>
They describe the same thing. <code>StructType</code> is built from Python objects and is good for generating schemas in code. A DDL string like <code>"id INT, name STRING"</code> is shorter and uses the same syntax as SQL <code>CREATE TABLE</code>.<br><br>

<b>🎤 4. When would you use <code>DecimalType</code> instead of <code>DoubleType</code>?</b><br>
For money and anything that needs exact precision. Doubles are binary floating point, so values like 0.1 can't be stored exactly and sums drift.<br><br>

<b>🎤 5. What happens when a cast fails?</b><br>
With ANSI mode on, which is the default in Spark 4 and on Databricks serverless, the query fails with an error such as <code>CAST_INVALID_INPUT</code>. With ANSI off it returns null. <code>try_cast</code> always returns null on failure, so it is the explicit way to get lenient behavior.<br><br>

<b>🎤 6. How do you access nested fields?</b><br>
Dot notation for structs (<code>address.city</code>), an index for arrays (<code>phones[0]</code>), and a key for maps (<code>attributes['tier']</code>).<br><br>

<b>🎤 7. Does <code>nullable=False</code> guarantee there are no nulls?</b><br>
It is enforced when you build a DataFrame from local data, and Delta tables enforce <code>NOT NULL</code> constraints on write. But nullability from some sources is only a hint, so production pipelines still run explicit null checks.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer reads a 2 TB CSV with <code>inferSchema=true</code> and notices the read is slow. What is the most likely reason?</b><br>
A. CSV files cannot be split across executors<br>
B. Spark makes an extra pass over the data to infer column types<br>
C. Inferred schemas are always stored as strings<br>
D. <code>inferSchema</code> disables partition pruning<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Inference requires reading the data before the actual job. Supplying a schema removes that pass.</details><br><br>

<b>Q2. Which data type should store a currency amount such as 1234.56?</b><br>
A. <code>DOUBLE</code><br>
B. <code>FLOAT</code><br>
C. <code>DECIMAL(12,2)</code><br>
D. <code>STRING</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> <code>DECIMAL</code> stores exact values. Floating-point types introduce rounding errors.</details><br><br>

<b>Q3. On a Databricks serverless notebook, <code>SELECT CAST('abc' AS INT)</code> fails. Which expression returns NULL instead?</b><br>
A. <code>CAST('abc' AS INT NULL)</code><br>
B. <code>try_cast('abc' AS INT)</code><br>
C. <code>safe_cast('abc' AS INT)</code><br>
D. <code>nvl(CAST('abc' AS INT), NULL)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> ANSI mode makes invalid casts fail. <code>try_cast</code> returns NULL on failure. <code>safe_cast</code> is BigQuery syntax, not Spark.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Define a schema for orders as a <b>DDL string</b>: <code>order_id INT NOT NULL</code>, <code>customer STRING</code>, <code>amount DECIMAL(10,2)</code>, <code>order_date DATE</code>, <code>items ARRAY&lt;STRING&gt;</code></li><li>Create a DataFrame with 3 orders using that schema and print its schema</li><li>Write the same schema as a <code>StructType</code> and check that both schemas are equal</li><li>Create a DataFrame of raw strings <code>("1","19.99","2024-02-01")</code>, <code>("2","abc","2024-02-02")</code> and convert it to <code>INT</code>, <code>DECIMAL(10,2)</code> and <code>DATE</code> without the job failing</li><li>Count how many amounts failed conversion</li><li>In one sentence: why are postcodes stored as strings?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own. Comparing approaches is where the learning happens.
</div>

In [0]:
# 1-2. DDL schema and a DataFrame that uses it
from datetime import date
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, DateType, ArrayType

orders_ddl = "order_id INT NOT NULL, customer STRING, amount DECIMAL(10,2), order_date DATE, items ARRAY<STRING>"
orders = spark.createDataFrame(
    [
        (1, "Amal", Decimal("19.99"), date(2024, 2, 1), ["pen"]),
        (2, "John", Decimal("250.00"), date(2024, 2, 2), ["desk", "lamp"]),
        (3, "Vivek", Decimal("5.50"), date(2024, 2, 3), []),
    ],
    orders_ddl,
)
orders.printSchema()

# 3. The same schema as a StructType, then compare
orders_struct = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer", StringType(), True),
    StructField("amount", DecimalType(10, 2), True),
    StructField("order_date", DateType(), True),
    StructField("items", ArrayType(StringType(), True), True),
])
print("Schemas equal:", orders.schema == orders_struct)

# 4. Lenient conversion with try_cast, so 'abc' becomes null instead of failing
raw_orders = spark.createDataFrame([("1", "19.99", "2024-02-01"), ("2", "abc", "2024-02-02")], "id STRING, amount STRING, d STRING")
clean = raw_orders.select(
    F.expr("try_cast(id AS INT)").alias("order_id"),
    F.expr("try_cast(amount AS DECIMAL(10,2))").alias("amount"),
    F.expr("try_cast(d AS DATE)").alias("order_date"),
)
clean.show()

# 5. Count failed conversions
print("Failed amounts:", clean.filter(F.col("amount").isNull()).count())

# 6. Postcodes are identifiers, not quantities: converting them to numbers drops leading zeros.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>A schema is a <code>StructType</code> of <code>StructField(name, type, nullable)</code>, or an equivalent DDL string</li><li>Prefer explicit schemas in production: faster, correct, and a documented contract</li><li><code>DECIMAL</code> for money, <code>DOUBLE</code> for measurements, <code>STRING</code> for identifiers</li><li>Nested types: <code>STRUCT</code> (dot notation), <code>ARRAY</code> (<code>[i]</code>), <code>MAP</code> (<code>[key]</code>)</li><li><code>cast</code> converts types. With ANSI mode on (serverless, Spark 4) an invalid cast <b>fails</b></li><li><code>try_cast</code> returns null on failure. Count those nulls as a data quality check</li><li><code>printSchema()</code>, <code>schema</code> and <code>dtypes</code> are metadata-only</li></ul>
</div>

| Task | Code |
|---|---|
| Schema as DDL | `"id INT NOT NULL, name STRING"` |
| Schema as object | `StructType([StructField("id", IntegerType(), False)])` |
| Cast | `F.col("x").cast("int")` |
| Safe cast | `F.expr("try_cast(x AS INT)")` |
| Nested field | `F.col("address.city")`, `F.col("tags")[0]`, `F.col("attrs")["k"]` |
| Schema to string | `df.schema.simpleString()` |

## Git commit

```
git add 01_spark_basics/04_schema_and_data_types.ipynb
git commit -m "add 04 schema and data types notebook"
```